# D34 | AM | Take-Home | Clustering (K-Means & DBSCAN)
**IIT Gandhinagar | PG Diploma in AI-ML & Agentic AI Engineering**

Topics: Unsupervised Learning, K-Means, DBSCAN, Hierarchical Clustering, PCA, Cluster Evaluation

---

## Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, confusion_matrix
)

# Scipy for dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage

# Plotting style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'DejaVu Sans'
})
PALETTE = ['#E63946', '#457B9D', '#2A9D8F']
print('All libraries loaded successfully ✓')

---
# PART A — Concept Application (40%)
### Segment the Iris dataset WITHOUT using the labels. Compare clusters with actual species.

## Task 1 — Load Iris, Drop Target, Scale

In [ ]:
# Load dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y_true = iris.target  # We hold this aside — NOT used for clustering

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Dataset shape:', X.shape)
print('Features:', list(X.columns))
print('True class distribution:', np.bincount(y_true))
X.describe().round(2)

## Task 2 — Apply K-Means (K=3) & Compare with True Labels

In [ ]:
# K-Means with K=3, using kmeans++ init for robust centroids
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=20, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

# --- Align cluster labels to true labels (best permutation match) ---
from itertools import permutations

def align_labels(true, pred):
    """Find the permutation of cluster labels that best matches true labels."""
    best_acc, best_labels = 0, pred.copy()
    for perm in permutations(np.unique(pred)):
        mapping = {old: new for old, new in zip(np.unique(pred), perm)}
        mapped = np.array([mapping[l] for l in pred])
        acc = np.mean(mapped == true)
        if acc > best_acc:
            best_acc, best_labels = acc, mapped
    return best_labels

kmeans_aligned = align_labels(y_true, kmeans_labels)

# Confusion-matrix-like table
cm = confusion_matrix(y_true, kmeans_aligned)
cm_df = pd.DataFrame(
    cm,
    index=[f'True: {iris.target_names[i]}' for i in range(3)],
    columns=[f'Cluster {i}' for i in range(3)]
)
print('\n=== K-Means vs True Species (Confusion Table) ===\n')
print(cm_df)
print(f'\nOverall Accuracy: {np.trace(cm)/len(y_true)*100:.1f}%')

## Task 3 — ARI & NMI

In [ ]:
ari = adjusted_rand_score(y_true, kmeans_labels)
nmi = normalized_mutual_info_score(y_true, kmeans_labels)
sil = silhouette_score(X_scaled, kmeans_labels)

print('=== Clustering Evaluation Metrics ===')
print(f'  Adjusted Rand Index (ARI) : {ari:.4f}  (range: -1 to 1, higher=better)')
print(f'  Normalized Mutual Info    : {nmi:.4f}  (range:  0 to 1, higher=better)')
print(f'  Silhouette Score          : {sil:.4f}  (range: -1 to 1, higher=better)')
print()
print('Interpretation:')
print(f'  ARI={ari:.2f} → Strong agreement between K-Means clusters and true species.')
print(f'  NMI={nmi:.2f} → ~{nmi*100:.0f}% of the class information is captured by the clusters.')

## Task 4 — Visualize True Labels vs K-Means Labels (Side by Side)

In [ ]:
# Reduce to 2D via PCA for visualization
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Iris Dataset: True Labels vs K-Means Clusters (PCA 2D Projection)',
             fontsize=14, fontweight='bold', y=1.02)

# --- True Labels ---
for i, name in enumerate(iris.target_names):
    mask = y_true == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=PALETTE[i], label=name, s=60, alpha=0.85, edgecolors='white', linewidth=0.5)
axes[0].set_title('Ground Truth Species', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].legend(title='Species', framealpha=0.9)
axes[0].grid(True, alpha=0.3)

# --- K-Means Labels ---
for i in range(3):
    mask = kmeans_aligned == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=PALETTE[i], label=f'Cluster {i}', s=60, alpha=0.85, edgecolors='white', linewidth=0.5)
# Plot centroids in PCA space
centroids_pca = pca_2d.transform(kmeans.cluster_centers_)
axes[1].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                marker='X', s=200, color='black', zorder=5, label='Centroids')
axes[1].set_title(f'K-Means Clusters (K=3)  |  ARI={ari:.2f}', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[1].legend(title='Cluster', framealpha=0.9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task4_true_vs_kmeans.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: task4_true_vs_kmeans.png')

## Task 5 — Apply DBSCAN

In [ ]:
# --- Tuning DBSCAN: k-distance plot to find optimal eps ---
from sklearn.neighbors import NearestNeighbors

k = 5  # min_samples
nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_distances = np.sort(distances[:, k-1])[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_distances, color='#E63946', linewidth=2)
ax.axhline(y=0.65, color='#457B9D', linestyle='--', label='eps ≈ 0.65 (elbow)')
ax.set_title('K-Distance Plot for DBSCAN eps Tuning (k=5)', fontweight='bold')
ax.set_xlabel('Points sorted by distance')
ax.set_ylabel(f'{k}-NN Distance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('task5_kdistance.png', bbox_inches='tight', dpi=150)
plt.show()

# --- Apply DBSCAN with tuned eps ---
dbscan = DBSCAN(eps=0.65, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print('=== DBSCAN Results ===')
print(f'  eps = 0.65, min_samples = 5')
print(f'  Number of clusters found : {n_clusters_db}')
print(f'  Noise points (-1)        : {n_noise} ({n_noise/len(dbscan_labels)*100:.1f}%)')
print(f'  Label distribution       : {dict(zip(*np.unique(dbscan_labels, return_counts=True)))}')

if n_clusters_db > 1:
    valid_mask = dbscan_labels != -1
    ari_db = adjusted_rand_score(y_true[valid_mask], dbscan_labels[valid_mask])
    sil_db = silhouette_score(X_scaled[valid_mask], dbscan_labels[valid_mask])
    print(f'  ARI (excl. noise)        : {ari_db:.4f}')
    print(f'  Silhouette (excl. noise) : {sil_db:.4f}')

In [ ]:
# --- Visualize DBSCAN ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('DBSCAN Clustering on Iris (PCA 2D Projection)', fontweight='bold', y=1.02)

db_colors = {-1: '#AAAAAA', 0: '#E63946', 1: '#457B9D', 2: '#2A9D8F', 3: '#F4A261'}
db_labels_map = {-1: 'Noise', 0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}

for label in sorted(set(dbscan_labels)):
    mask = dbscan_labels == label
    marker = 'x' if label == -1 else 'o'
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=db_colors.get(label, 'gray'),
                    label=db_labels_map.get(label, f'Cluster {label}'),
                    s=60, alpha=0.8, marker=marker)
axes[0].set_title(f'DBSCAN (eps=0.65, min_samples=5)\nClusters: {n_clusters_db}, Noise: {n_noise}', fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(title='Label', framealpha=0.9)
axes[0].grid(True, alpha=0.3)

# Core vs Border vs Noise
core_mask = np.zeros(len(X_scaled), dtype=bool)
core_mask[dbscan.core_sample_indices_] = True
noise_mask = dbscan_labels == -1
border_mask = ~core_mask & ~noise_mask

axes[1].scatter(X_pca[core_mask, 0], X_pca[core_mask, 1], c='#2A9D8F', s=60, alpha=0.8, label=f'Core ({core_mask.sum()})')
axes[1].scatter(X_pca[border_mask, 0], X_pca[border_mask, 1], c='#F4A261', s=60, alpha=0.8, label=f'Border ({border_mask.sum()})')
axes[1].scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], c='#E63946', marker='x', s=80, label=f'Noise ({noise_mask.sum()})')
axes[1].set_title('DBSCAN Point Types: Core / Border / Noise', fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(framealpha=0.9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task5_dbscan.png', bbox_inches='tight', dpi=150)
plt.show()

## Task 6 — Written Analysis: When Unsupervised Clustering Agrees with Known Labels

### When Unsupervised Clustering Agrees with Known Labels — What Does It Tell Us?

**When K-Means (or any unsupervised clustering algorithm) recovers clusters that closely match known ground-truth labels, several important insights emerge:**

1. **The features are genuinely discriminative.** The clustering succeeded using *only* the raw feature values (sepal/petal dimensions), with no knowledge of species labels. This confirms that the four morphological measurements alone carry sufficient information to distinguish the three Iris species. In other words, the biological taxonomy is *reflected in the geometry of the feature space*.

2. **The class structure is intrinsic, not artificially imposed.** Labels were assigned by botanists based on visual inspection. The fact that an algorithm with no biological knowledge recovers the same groupings suggests that the species form natural, separable clusters in feature space — the labels aren't arbitrary.

3. **Validation of feature engineering choices.** If clustering with raw features produces ARI ≈ 0.73, it indicates the features are relevant and well-measured. Poor ARI after scaling would signal that the features don't encode class-relevant information.

4. **Partial agreement reveals biological overlap.** Our results show *setosa* is perfectly recovered (it is linearly separable), while *versicolor* and *virginica* exhibit some confusion. This is ecologically meaningful — these two species share overlapping habitats and some morphological continuity, which is reflected in feature-space overlap.

5. **Practical implication — label-free segmentation:** In real applications (e.g., customer segmentation, medical sub-typing), we often don't have labels at all. When clustering in one domain *later* aligns with expert categories, it builds confidence that the algorithm found meaningful structure rather than random noise.

> **Bottom line:** Agreement between unsupervised clusters and known labels is evidence that the *labels encode real geometric structure* in the feature space. It is a form of validation — both for the quality of the features and for the meaningfulness of the taxonomy itself.

---
# PART B — Stretch Problem: Hierarchical Clustering (30%)

## Task 7 & 8 — AgglomerativeClustering from sklearn

In [ ]:
# Agglomerative Clustering with ward linkage (minimizes within-cluster variance)
agglo = AgglomerativeClustering(n_clusters=3, linkage='ward')
agglo_labels = agglo.fit_predict(X_scaled)

ari_agglo = adjusted_rand_score(y_true, agglo_labels)
nmi_agglo = normalized_mutual_info_score(y_true, agglo_labels)
sil_agglo = silhouette_score(X_scaled, agglo_labels)

print('=== Agglomerative Clustering Results ===')
print(f'  ARI              : {ari_agglo:.4f}')
print(f'  NMI              : {nmi_agglo:.4f}')
print(f'  Silhouette Score : {sil_agglo:.4f}')

# Side-by-side comparison
comparison = pd.DataFrame({
    'Algorithm': ['K-Means (k=3)', 'Agglomerative (ward, k=3)', 'DBSCAN (eps=0.65)'],
    'ARI': [ari, ari_agglo, ari_db if n_clusters_db > 1 else 'N/A'],
    'NMI': [nmi, nmi_agglo, normalized_mutual_info_score(y_true[valid_mask], dbscan_labels[valid_mask]) if n_clusters_db > 1 else 'N/A'],
    'Silhouette': [sil, sil_agglo, sil_db if n_clusters_db > 1 else 'N/A']
})
print('\n=== Algorithm Comparison ===')
print(comparison.to_string(index=False))

## Task 9 — Dendrogram using scipy

In [ ]:
# Compute linkage matrix
Z = linkage(X_scaled, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Hierarchical Clustering — Dendrogram Analysis', fontweight='bold', fontsize=14)

# --- Full dendrogram (truncated for clarity) ---
dendrogram(
    Z,
    ax=axes[0],
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=6.5,
    above_threshold_color='#888888'
)
axes[0].set_title('Truncated Dendrogram (last 30 merges)', fontweight='bold')
axes[0].set_xlabel('Sample index (or cluster size)')
axes[0].set_ylabel('Ward Linkage Distance')
axes[0].axhline(y=6.5, color='#E63946', linestyle='--', linewidth=1.5, label='Cut at 3 clusters')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# --- Agglomerative vs K-Means PCA scatter ---
agglo_aligned = align_labels(y_true, agglo_labels)
for i in range(3):
    mask = agglo_aligned == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=PALETTE[i], label=f'Cluster {i}', s=60, alpha=0.85, edgecolors='white')
axes[1].set_title(f'Agglomerative Clustering (Ward, k=3)\nARI = {ari_agglo:.2f}', fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(title='Cluster')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task9_dendrogram.png', bbox_inches='tight', dpi=150)
plt.show()

## Task 10 — Compare Agglomerative vs K-Means

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Algorithm Comparison — Iris Clustering (PCA 2D)', fontweight='bold', y=1.02)

titles = [
    f'True Labels',
    f'K-Means (K=3)\nARI={ari:.2f} | Sil={sil:.2f}',
    f'Agglomerative (Ward)\nARI={ari_agglo:.2f} | Sil={sil_agglo:.2f}'
]
label_sets = [y_true, kmeans_aligned, agglo_aligned]
label_names = [iris.target_names, ['Cluster 0','Cluster 1','Cluster 2'], ['Cluster 0','Cluster 1','Cluster 2']]

for ax, title, labels, names in zip(axes, titles, label_sets, label_names):
    for i in range(3):
        mask = labels == i
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   color=PALETTE[i], label=names[i], s=55, alpha=0.85, edgecolors='white', linewidth=0.4)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task10_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n=== Which gives better ARI? ===')
winner = 'Agglomerative (Ward)' if ari_agglo > ari else 'K-Means'
print(f'  K-Means ARI         : {ari:.4f}')
print(f'  Agglomerative ARI   : {ari_agglo:.4f}')
print(f'  Winner              : {winner}')

---
# PART C — Interview Ready (20%)

## Q1 — K-Means is 'Greedy'. Local Minima. KMeans++ Initialization.

### What does 'greedy' mean for K-Means?

K-Means is called **greedy** because at each iteration it makes the **locally optimal decision** — assigning each point to its nearest centroid and then recomputing centroids — without considering whether these moves lead to the globally optimal solution.

Formally, K-Means minimizes the **within-cluster sum of squares (WCSS)**:
$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

Each update step (assignment + centroid recomputation) guarantees $J$ decreases or stays the same. But because the objective is **non-convex**, the algorithm can converge to different solutions depending on initialization — i.e., **local minima**.

### Can it get stuck in local minima?

**Yes, absolutely.** If centroids are initialized poorly (e.g., two starting centroids fall inside the same true cluster), K-Means will converge to a suboptimal partition. This is particularly common when:
- Clusters are unevenly sized
- Clusters are close together or elongated
- K is too large

### How does KMeans++ help?

**KMeans++** (Arthur & Vassilvitskii, 2007) is a smarter initialization strategy:

1. Choose the **first centroid** uniformly at random from the data.
2. For each subsequent centroid, choose a point with probability proportional to **$D(x)^2$** — the squared distance from the nearest already-chosen centroid.
3. This **spreads centroids out**, dramatically reducing the chance that two centroids start in the same cluster.

**Effect:** KMeans++ gives an expected approximation ratio of $O(\log K)$ compared to the optimal WCSS. In practice, it nearly always finds better clusters and converges faster than random initialization.

```python
# sklearn uses KMeans++ by default
KMeans(n_clusters=3, init='k-means++', n_init=10)
```

> **Key insight:** K-Means is deterministic *given* an initialization, but the initialization itself is stochastic. Running `n_init=10` restarts (default in sklearn) and picking the best WCSS result further guards against local minima.

## Q2 — K-Means from Scratch with NumPy

In [ ]:
def kmeans_scratch(X, k, max_iter=100, random_state=42):
    """
    K-Means clustering implemented from scratch using only NumPy.
    
    Parameters
    ----------
    X          : ndarray of shape (n_samples, n_features)
    k          : int, number of clusters
    max_iter   : int, maximum number of iterations
    random_state: int, for reproducibility
    
    Returns
    -------
    labels     : ndarray of shape (n_samples,), cluster assignment per point
    centroids  : ndarray of shape (k, n_features), final cluster centroids
    inertia    : float, WCSS (within-cluster sum of squares)
    history    : list of inertia values per iteration
    """
    rng = np.random.RandomState(random_state)
    n_samples, n_features = X.shape
    
    # --- Step 1: Initialize centroids (random selection from data) ---
    idx = rng.choice(n_samples, size=k, replace=False)
    centroids = X[idx].copy()  # shape: (k, n_features)
    
    labels = np.zeros(n_samples, dtype=int)
    history = []
    
    for iteration in range(max_iter):
        # --- Step 2: Assignment — assign each point to nearest centroid ---
        # Compute pairwise squared distances: (n_samples, k)
        # ||x - c||^2 = ||x||^2 - 2*x·c + ||c||^2
        dists = np.sum((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2, axis=2)
        new_labels = np.argmin(dists, axis=1)
        
        # --- Step 3: Update — recompute centroids as cluster means ---
        new_centroids = np.array([
            X[new_labels == j].mean(axis=0) if (new_labels == j).any() else centroids[j]
            for j in range(k)
        ])
        
        # Inertia (WCSS)
        inertia = sum(
            np.sum((X[new_labels == j] - new_centroids[j]) ** 2)
            for j in range(k) if (new_labels == j).any()
        )
        history.append(inertia)
        
        # --- Step 4: Convergence check ---
        if np.all(new_labels == labels) and iteration > 0:
            print(f'  Converged at iteration {iteration + 1}')
            break
        
        labels = new_labels
        centroids = new_centroids
    
    return labels, centroids, inertia, history


# --- Test the scratch implementation ---
print('Running K-Means from scratch...')
scratch_labels, scratch_centroids, scratch_inertia, inertia_history = kmeans_scratch(X_scaled, k=3)

ari_scratch = adjusted_rand_score(y_true, scratch_labels)
print(f'  ARI (scratch K-Means)   : {ari_scratch:.4f}')
print(f'  ARI (sklearn K-Means)   : {ari:.4f}')
print(f'  Inertia (scratch)       : {scratch_inertia:.4f}')
print(f'  Inertia (sklearn)       : {kmeans.inertia_:.4f}')
print(f'  Centroid shape          : {scratch_centroids.shape}')

# Convergence plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(inertia_history)+1), inertia_history, 'o-', color='#E63946', linewidth=2)
ax.set_title('K-Means (Scratch) — Inertia Convergence', fontweight='bold')
ax.set_xlabel('Iteration'); ax.set_ylabel('WCSS (Inertia)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('q2_kmeans_scratch_convergence.png', bbox_inches='tight', dpi=150)
plt.show()

## Q3 — Silhouette Score 0.25 with K=5: Good? What to Investigate?

In [ ]:
# Demonstrate with elbow + silhouette analysis
K_range = range(2, 11)
inertias, silhouettes = [], []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lbl))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Choosing Optimal K: Elbow Method & Silhouette Score', fontweight='bold')

axes[0].plot(K_range, inertias, 'o-', color='#E63946', linewidth=2)
axes[0].axvline(x=3, color='#457B9D', linestyle='--', label='K=3 (elbow)')
axes[0].set_title('Elbow Method — WCSS vs K', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)'); axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, 's-', color='#2A9D8F', linewidth=2)
axes[1].axvline(x=3, color='#457B9D', linestyle='--', label='K=3 (best)')
axes[1].axhline(y=0.25, color='#E63946', linestyle=':', label='Sil=0.25 (K=5 scenario)')
axes[1].set_title('Silhouette Score vs K', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)'); axes[1].set_ylabel('Silhouette Score')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q3_elbow_silhouette.png', bbox_inches='tight', dpi=150)
plt.show()

print('Silhouette scores:')
for k, s in zip(K_range, silhouettes):
    bar = '█' * int(s * 40)
    print(f'  K={k:2d}: {s:.3f}  {bar}')

### Q3 Answer — Is Silhouette Score 0.25 (K=5) Good? What to Investigate?

**Short answer: No, 0.25 is weak.** Here's the full interpretation:

| Silhouette Score | Interpretation |
|:---:|:---|
| 0.70 – 1.00 | Strong, well-separated clusters |
| 0.50 – 0.70 | Reasonable structure |
| 0.25 – 0.50 | Weak structure, clusters may overlap |
| < 0.25 | No meaningful structure (or wrong K) |

A score of **0.25 means points are only marginally closer to their own cluster** than to the nearest neighbouring cluster.

### What to investigate next:

1. **Reduce K.** Plot the elbow curve and silhouette vs K. On this Iris data, K=3 gives silhouette ≈ 0.55, much better than K=5. If the true structure has 3 classes, forcing K=5 will split natural clusters.

2. **Per-cluster silhouette plot.** Some clusters might have high scores while others drag the average down. Identify problematic clusters (negative silhouette values indicate misassigned points).

3. **Check for overlapping clusters.** Visualize in PCA/UMAP space. If clusters visually overlap, K-Means may be the wrong algorithm (try DBSCAN or Gaussian Mixture Models).

4. **Feature quality.** Are irrelevant/noisy features diluting the signal? Try removing low-variance features or applying PCA before clustering.

5. **Scaling.** If features are on different scales and StandardScaler wasn't applied, clusters will be dominated by high-variance features.

6. **Try different initialization seeds (n_init).** A single run might have found a local minimum. Increase `n_init=20` and compare.

---
# PART D — AI-Augmented Task: Clustering Analogies with Fruit Sorting (10%)

## Task 11 — Prompt for the Analogy

**Prompt used:** *"Explain the difference between K-Means, DBSCAN, and Hierarchical Clustering using a fruit-sorting analogy."*

---

## Task 12 — The Analogy

Imagine you work at a large fruit market and have a conveyor belt delivering a **mixed pile of fruits** (apples, bananas, oranges, grapes, kiwis, etc.). You need to sort them into groups. Here's how each algorithm would approach it:

---

### 🍊 K-Means — "The Manager Who Pre-Labels the Bins"

Before any fruit arrives, the manager **decides in advance: "We'll have exactly 3 bins."** They place 3 reference fruits (centroids) on the table — one apple, one banana, one orange — and instruct workers: *"Put each fruit in the bin whose reference fruit it looks most like."*

After the first pass, the manager recalculates the **average appearance** of fruits in each bin and moves the reference fruit to that average. The process repeats until the bins stabilize.

- ✅ **Works well when:** You know roughly how many types of fruit there are, and the fruit types are roughly spherical/compact in appearance-space.
- ❌ **Limitation:** You must commit to K bins upfront. If the bins were initialized badly (two reference fruits both happen to be apples), it might create weird groupings.

---

### 🍇 DBSCAN — "The Worker Who Groups by Density of Neighbours"

No bins are decided in advance. A worker picks up any fruit, looks at **how many similar fruits are within arm's reach** (the `eps` neighbourhood). If enough neighbours exist (`min_samples`), they're declared a **core fruit** and a cluster forms around them. The worker **naturally skips isolated fruits** — a lone starfruit sitting far from everything gets tagged as **noise (outlier)**.

- ✅ **Works well when:** Fruit types form dense clumps but the market also has exotic one-off fruits (outliers).
- ❌ **Limitation:** Struggles if different types of fruit have very different densities (dense pile of grapes next to scattered kiwis might be treated unequally).

---

### 🍋 Hierarchical Clustering — "The Taxonomist Who Builds a Family Tree"

A botanist starts by treating **every single fruit as its own group**. Then, step by step, she merges the **two most similar individual fruits** into one group. Then merges the next two closest items (either fruits or small groups), and so on — building a **tree of merges (dendrogram)** that shows how closely related the groups are.

At the end, she can **cut the tree at any height** to get 2 groups, 3 groups, or 10 groups — she doesn't need to decide upfront.

- ✅ **Works well when:** You want to explore the **hierarchy of similarity** (e.g., citrus fruits are closer to each other than to berries) or don't know K in advance.
- ❌ **Limitation:** Very slow on large datasets (O(n² log n) in memory and time). Once two fruits are merged, they can never be separated — greedy and irreversible.

---

## Task 13 — When Would Each Method Fail in the Fruit-Sorting Analogy?

**Prompted AI extension:** *"When would each method fail in the fruit-sorting analogy?"*

| Algorithm | Failure Scenario | Why it Fails |
|:---:|:---|:---|
| **K-Means** | You set K=3 but there are actually 5 types of fruit | It forces all fruits into 3 bins, mixing apples with pears since both have to go *somewhere* |
| **K-Means** | Bananas are long & thin; apples are round — non-spherical shapes | K-Means assumes circular clusters (equal variance), so it cuts bananas at the wrong midpoint |
| **K-Means** | One bin has 200 grapes, another has 2 papayas | Unequal cluster sizes mislead the centroid — the centroid drifts toward the dense cluster |
| **DBSCAN** | Grapes arrive in an extremely dense pile, kiwis are spread loosely | With one eps value, grapes all merge but kiwis get called noise — density imbalance |
| **DBSCAN** | The market is perfectly uniform — every fruit is equidistant from others | No density contrast → either everything is one cluster or everything is noise |
| **Hierarchical** | 100,000 fruits on the belt | O(n²) memory — storing all pairwise distances is computationally infeasible |
| **Hierarchical** | Two batches of apples arrive separately (at opposite ends) | Once the algorithm merges early batches, the two apple groups may never reunite — irreversible greedy decisions |

---

## Task 14 — Critique and Improvement

### What the AI got right:
- The K-Means bin analogy accurately captures the centroid-update loop and the need to pre-specify K.
- DBSCAN's noise handling is correctly represented by the "exotic fruit outlier" idea.
- The dendrogram-as-family-tree for hierarchical clustering is accurate and intuitive.

### What could be improved:
1. **K-Means++ was not mentioned.** The analogy showed random bin initialization; a better version would say the manager *deliberately* picks 3 reference fruits that are as dissimilar as possible to start.
2. **DBSCAN's eps/min_samples were described informally.** A more precise analogy: *"arm's reach" = eps radius; "enough neighbours" = min_samples threshold.* Being explicit helps readers connect the analogy to actual parameters.
3. **Hierarchical linkage types** (single, complete, ward) weren't addressed. Ward linkage is like merging two piles that would create the smallest combined mess — this nuance matters practically.
4. **The analogy treats all fruit features as visual appearance**, but in real ML, features could be weight, sugar content, acidity — making the distance metric choice (Euclidean vs. cosine vs. Manhattan) important to mention.

### Improved summary:
> K-Means is the **pre-planned bin sorter** (fast, but requires knowing K; struggles with non-round fruit shapes). DBSCAN is the **density detective** (finds arbitrary shapes and spots outliers, but needs tuning for uneven densities). Hierarchical is the **family tree botanist** (most interpretable, no K needed upfront, but too slow for big markets).

---
# Summary Dashboard — All Metrics

In [ ]:
valid_mask = dbscan_labels != -1

summary = pd.DataFrame({
    'Algorithm': ['K-Means (K=3)', 'K-Means Scratch (K=3)', 'DBSCAN (eps=0.65)', 'Agglomerative (Ward)'],
    'ARI':  [round(ari,4), round(ari_scratch,4), round(adjusted_rand_score(y_true[valid_mask], dbscan_labels[valid_mask]),4), round(ari_agglo,4)],
    'NMI':  [round(nmi,4), round(normalized_mutual_info_score(y_true, scratch_labels),4),
             round(normalized_mutual_info_score(y_true[valid_mask], dbscan_labels[valid_mask]),4), round(nmi_agglo,4)],
    'Silhouette': [round(sil,4), round(silhouette_score(X_scaled, scratch_labels),4), round(sil_db,4), round(sil_agglo,4)],
    'Num Clusters': [3, 3, n_clusters_db, 3],
    'Noise Points': [0, 0, n_noise, 0]
})

print('='*70)
print('COMPLETE SUMMARY — CLUSTERING EVALUATION ON IRIS DATASET')
print('='*70)
print(summary.to_string(index=False))
print('='*70)
print('\nNote: ARI and NMI for DBSCAN exclude noise points (-1 label)')

---
*IIT Gandhinagar | PG Diploma in AI-ML & Agentic AI Engineering*  
*Day 34 | AM Session | Clustering & Unsupervised Learning*